# 1. Introduction
Notebook reproductible pour explorer, nettoyer et joindre customers, products et sales. Les sources dans `data/` ne sont jamais modifiees.

# 2. Importation des bibliothèques

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.extraction import find_source_files, load_sources
from src.cleaning import clean_customers, clean_products, clean_sales
from src.joining import relationship_report, build_dataset_clean

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


BASE_DIR=PROJECT_ROOT
DATA_DIR=BASE_DIR/'data/raw' 
OUTPUT_DIR=BASE_DIR/'data/processed'
OUTPUT_DIR.mkdir(exist_ok=True)

# 3. Chargement des données
Les trois CSV sont detectes automatiquement a partir de leur nom.

In [2]:
paths=find_source_files(DATA_DIR)
raw=load_sources(DATA_DIR)

for name, df in raw.items():
 print(name, paths[name].name, df.shape, df.columns.tolist()); display(df.head())

customers_data customers_data.csv (5, 7) ['Customer_ID', 'Name', 'Age', 'Gender', 'Location', 'Join_Date', 'Total_Spent']


,Customer_ID,Name,Age,Gender,Location,Join_Date,Total_Spent
0,2001,Alice,28,Female,New York,2022-05-10,500.0
1,2002,Bob,35,Male,Los Angeles,2022-06-15,750.0
2,2003,Charlie,22,Male,Chicago,2022-07-20,300.0
3,2004,Diana,30,Female,Houston,2022-08-25,600.0
4,2005,Eva,27,Female,Phoenix,2022-09-30,450.0


products_data products_data.csv (5, 5) ['Product_ID', 'Product_Name', 'Category', 'Price', 'Brand']


,Product_ID,Product_Name,Category,Price,Brand
0,101,T-shirt,Clothing,25.0,Brand A
1,102,Jeans,Clothing,75.0,Brand B
2,103,Sneakers,Footwear,30.0,Brand C
3,104,Jacket,Outerwear,120.0,Brand D
4,105,Hat,Accessories,22.5,Brand E


sales_data sales_data.csv (5, 7) ['Sale_ID', 'Product_ID', 'Customer_ID', 'Date', 'Quantity', 'Sale_Price', 'Channel']


,Sale_ID,Product_ID,Customer_ID,Date,Quantity,Sale_Price,Channel
0,1,101,2001,2023-01-15,2,50.0,Online
1,2,102,2002,2023-01-16,1,75.0,In-Store
2,3,103,2001,2023-01-17,3,30.0,Online
3,4,104,2003,2023-01-18,1,120.0,In-Store
4,5,105,2004,2023-01-19,2,45.0,Online


# 4. Exploration des données
## 4.1 Customers
## 4.2 Products
## 4.3 Sales
Les controles suivants affichent tailles, types, statistiques, valeurs manquantes, doublons, unicite des identifiants et anomalies numeriques. Les cles candidates observees sont `Customer_ID`, `Product_ID` et `Sale_ID`.

In [3]:
for name, df in raw.items():
 print('\n###', name); 
 print(df.dtypes); display(df.describe(include='all'))
 display(pd.DataFrame({'missing':df.isna().sum(),'unique':df.nunique()}))
 print('Doublons exacts:',df.duplicated().sum())

print('Customer_ID dupliques:', raw['customers_data']['Customer_ID'].duplicated().sum())
print('Product_ID dupliques:', raw['products_data']['Product_ID'].duplicated().sum())

sales_dates=pd.to_datetime(raw['sales_data']['Date'],errors='coerce'); print('Periode ventes:',sales_dates.min(),sales_dates.max())

print('Prix <= 0:',(pd.to_numeric(raw['products_data']['Price'],errors='coerce')<=0).sum())
print('Quantites/Montants negatifs:',(pd.to_numeric(raw['sales_data']['Quantity'],errors='coerce')<0).sum(),(pd.to_numeric(raw['sales_data']['Sale_Price'],errors='coerce')<0).sum())


### customers_data
Customer_ID      int64
Name            object
Age              int64
Gender          object
Location        object
Join_Date       object
Total_Spent    float64
dtype: object


,Customer_ID,Name,Age,Gender,Location,Join_Date,Total_Spent
count,5.000000,5,5.000000,5,5,5,5.000000
unique,NaN,5,NaN,2,5,5,NaN
top,NaN,Alice,NaN,Female,New York,2022-05-10,NaN
freq,NaN,1,NaN,3,1,1,NaN
mean,2003.000000,NaN,28.400000,NaN,NaN,NaN,520.000000
std,1.581139,NaN,4.722288,NaN,NaN,NaN,168.077363
min,2001.000000,NaN,22.000000,NaN,NaN,NaN,300.000000
25%,2002.000000,NaN,27.000000,NaN,NaN,NaN,450.000000
50%,2003.000000,NaN,28.000000,NaN,NaN,NaN,500.000000
75%,2004.000000,NaN,30.000000,NaN,NaN,NaN,600.000000


,missing,unique
Customer_ID,0,5
Name,0,5
Age,0,5
Gender,0,2
Location,0,5
Join_Date,0,5
Total_Spent,0,5


Doublons exacts: 0

### products_data
Product_ID        int64
Product_Name     object
Category         object
Price           float64
Brand            object
dtype: object


,Product_ID,Product_Name,Category,Price,Brand
count,5.000000,5,5,5.000000,5
unique,NaN,5,4,NaN,5
top,NaN,T-shirt,Clothing,NaN,Brand A
freq,NaN,1,2,NaN,1
mean,103.000000,NaN,NaN,54.500000,NaN
std,1.581139,NaN,NaN,42.441136,NaN
min,101.000000,NaN,NaN,22.500000,NaN
25%,102.000000,NaN,NaN,25.000000,NaN
50%,103.000000,NaN,NaN,30.000000,NaN
75%,104.000000,NaN,NaN,75.000000,NaN


,missing,unique
Product_ID,0,5
Product_Name,0,5
Category,0,4
Price,0,5
Brand,0,5


Doublons exacts: 0

### sales_data
Sale_ID          int64
Product_ID       int64
Customer_ID      int64
Date            object
Quantity         int64
Sale_Price     float64
Channel         object
dtype: object


,Sale_ID,Product_ID,Customer_ID,Date,Quantity,Sale_Price,Channel
count,5.000000,5.000000,5.00000,5,5.00000,5.000000,5
unique,NaN,NaN,NaN,5,NaN,NaN,2
top,NaN,NaN,NaN,2023-01-15,NaN,NaN,Online
freq,NaN,NaN,NaN,1,NaN,NaN,3
mean,3.000000,103.000000,2002.20000,NaN,1.80000,64.000000,NaN
std,1.581139,1.581139,1.30384,NaN,0.83666,35.249113,NaN
min,1.000000,101.000000,2001.00000,NaN,1.00000,30.000000,NaN
25%,2.000000,102.000000,2001.00000,NaN,1.00000,45.000000,NaN
50%,3.000000,103.000000,2002.00000,NaN,2.00000,50.000000,NaN
75%,4.000000,104.000000,2003.00000,NaN,2.00000,75.000000,NaN


,missing,unique
Sale_ID,0,5
Product_ID,0,5
Customer_ID,0,4
Date,0,5
Quantity,0,3
Sale_Price,0,5
Channel,0,2


Doublons exacts: 0
Customer_ID dupliques: 0
Product_ID dupliques: 0
Periode ventes: 2023-01-15 00:00:00 2023-01-19 00:00:00
Prix <= 0: 0
Quantites/Montants negatifs: 0 0


# 5. Nettoyage des données
## 5.1 Customers
Les espaces texte sont retires ; seuls les doublons strictement identiques sont supprimes.
## 5.2 Products
Les prix sont convertis en numerique ; les prix absents, nuls ou negatifs sont invalides et sont retires.
## 5.3 Sales
Les dates, quantites et montants sont convertis. Les lignes avec date, quantite ou montant invalide sont retirees, car elles empechent une analyse fiable.

In [4]:
cleaned={
    'customers_data':clean_customers(raw['customers_data']),
    'products_data':clean_products(raw['products_data']),
    'sales_data':clean_sales(raw['sales_data'])
    }

for name, df in cleaned.items(): 
    print(name,'avant=',len(raw[name]),'apres=',len(df),'supprimees=',len(raw[name])-len(df))
    display(df.head())

customers_data avant= 5 apres= 5 supprimees= 0


,Customer_ID,Name,Age,Gender,Location,Join_Date,Total_Spent
0,2001,Alice,28,Female,New York,2022-05-10,500.0
1,2002,Bob,35,Male,Los Angeles,2022-06-15,750.0
2,2003,Charlie,22,Male,Chicago,2022-07-20,300.0
3,2004,Diana,30,Female,Houston,2022-08-25,600.0
4,2005,Eva,27,Female,Phoenix,2022-09-30,450.0


products_data avant= 5 apres= 5 supprimees= 0


,Product_ID,Product_Name,Category,Price,Brand
0,101,T-shirt,Clothing,25.0,Brand A
1,102,Jeans,Clothing,75.0,Brand B
2,103,Sneakers,Footwear,30.0,Brand C
3,104,Jacket,Outerwear,120.0,Brand D
4,105,Hat,Accessories,22.5,Brand E


sales_data avant= 5 apres= 5 supprimees= 0


,Sale_ID,Product_ID,Customer_ID,Date,Quantity,Sale_Price,Channel
0,1,101,2001,2023-01-15,2,50.0,Online
1,2,102,2002,2023-01-16,1,75.0,In-Store
2,3,103,2001,2023-01-17,3,30.0,Online
3,4,104,2003,2023-01-18,1,120.0,In-Store
4,5,105,2004,2023-01-19,2,45.0,Online


# 6. Visualisation
Ces graphiques presentent les profils clients, le catalogue produit et le chiffre d'affaires dans le temps.

In [5]:
fig,ax=plt.subplots(1,3,figsize=(15,4))
cleaned['customers_data']['Age'].plot.hist(bins=10,ax=ax[0],title='Distribution des ages'); ax[0].set_xlabel('Age')
cleaned['products_data']['Category'].value_counts().plot.bar(ax=ax[1],title='Produits par categorie'); ax[1].set_ylabel('Nombre')
daily=cleaned['sales_data'].assign(Revenue=lambda x:x.Quantity*x.Sale_Price).groupby('Date').Revenue.sum(); daily.plot(ax=ax[2],marker='o',title="Chiffre d'affaires"); ax[2].set_ylabel('CA'); plt.tight_layout(); plt.show()

<Figure size 1500x400 with 3 Axes>

# 7. Jointure des tables
La relation verifiee est `customers -> sales <- products`. `sales` est la table centrale. Les jointures `many_to_one` protegent contre la multiplication de lignes.

In [6]:
relations=relationship_report(cleaned['customers_data'],cleaned['products_data'],cleaned['sales_data']); display(pd.DataFrame([relations]))
dataset_clean=build_dataset_clean(cleaned['customers_data'],cleaned['products_data'],cleaned['sales_data'])
display(dataset_clean.head())

,customer_primary_key_duplicates,product_primary_key_duplicates,sales_unknown_customers,sales_unknown_products
0,0,0,0,0


,Sale_ID,Product_ID,Customer_ID,Date,Quantity,Sale_Price,Channel,Product_Name,Category,Price,Brand,Name,Age,Gender,Location,Join_Date,Total_Spent,Revenue
0,1,101,2001,2023-01-15,2,50.0,Online,T-shirt,Clothing,25.0,Brand A,Alice,28,Female,New York,2022-05-10,500.0,100.0
1,2,102,2002,2023-01-16,1,75.0,In-Store,Jeans,Clothing,75.0,Brand B,Bob,35,Male,Los Angeles,2022-06-15,750.0,75.0
2,3,103,2001,2023-01-17,3,30.0,Online,Sneakers,Footwear,30.0,Brand C,Alice,28,Female,New York,2022-05-10,500.0,90.0
3,4,104,2003,2023-01-18,1,120.0,In-Store,Jacket,Outerwear,120.0,Brand D,Charlie,22,Male,Chicago,2022-07-20,300.0,120.0
4,5,105,2004,2023-01-19,2,45.0,Online,Hat,Accessories,22.5,Brand E,Diana,30,Female,Houston,2022-08-25,600.0,90.0


# 8. Contrôle de qualité du dataset final

In [7]:
quality={
    'sales_before_join':len(cleaned['sales_data']),
    'dataset_rows':len(dataset_clean),
    'duplicates':dataset_clean.duplicated().sum(),
    'missing':dataset_clean.isna().sum().sum(),**relations}
display(pd.DataFrame([quality]))
assert len(dataset_clean)==len(cleaned['sales_data'])

,sales_before_join,dataset_rows,duplicates,missing,customer_primary_key_duplicates,product_primary_key_duplicates,sales_unknown_customers,sales_unknown_products
0,5,5,0,0,0,0,0,0


# 9. Résumé / conclusions

In [8]:
print(f"Clients: {len(cleaned['customers_data'])}; Produits: {len(cleaned['products_data'])}; Ventes: {len(cleaned['sales_data'])}")
print('Suppressions:',{n:len(raw[n])-len(cleaned[n]) for n in cleaned})
print('Cles: Customer_ID et Product_ID vers sales; lignes finales:',len(dataset_clean))
print('CA:',dataset_clean.Revenue.sum()); print('Anomalies relationnelles:',relations)

Clients: 5; Produits: 5; Ventes: 5
Suppressions: {'customers_data': 0, 'products_data': 0, 'sales_data': 0}
Cles: Customer_ID et Product_ID vers sales; lignes finales: 5
CA: 475.0
Anomalies relationnelles: {'customer_primary_key_duplicates': 0, 'product_primary_key_duplicates': 0, 'sales_unknown_customers': 0, 'sales_unknown_products': 0}


# 10. Export du dataset propre

In [9]:
for name,df in cleaned.items(): 
    df.to_csv(OUTPUT_DIR/f'{name}_clean.csv',index=False,encoding='utf-8')
    
dataset_clean.to_csv(OUTPUT_DIR/'dataset_data_final.csv',index=False,encoding='utf-8')
print(list(OUTPUT_DIR.iterdir()))
for path in [OUTPUT_DIR/'customers_data_clean.csv',OUTPUT_DIR/'products_data_clean.csv',OUTPUT_DIR/'sales_data_clean.csv',OUTPUT_DIR/'dataset_data_final.csv']: 
    assert path.exists()
    print(path.resolve())

[PosixPath('/media/Celesy20/M1 OCC/ML & DL/Examen M1/Segmentation-Client-Marketing-IA/data/processed/customers_data_clean.csv'), PosixPath('/media/Celesy20/M1 OCC/ML & DL/Examen M1/Segmentation-Client-Marketing-IA/data/processed/dataset_data_final.csv'), PosixPath('/media/Celesy20/M1 OCC/ML & DL/Examen M1/Segmentation-Client-Marketing-IA/data/processed/products_data_clean.csv'), PosixPath('/media/Celesy20/M1 OCC/ML & DL/Examen M1/Segmentation-Client-Marketing-IA/data/processed/sales_data_clean.csv')]
/media/Celesy20/M1 OCC/ML & DL/Examen M1/Segmentation-Client-Marketing-IA/data/processed/customers_data_clean.csv
/media/Celesy20/M1 OCC/ML & DL/Examen M1/Segmentation-Client-Marketing-IA/data/processed/products_data_clean.csv
/media/Celesy20/M1 OCC/ML & DL/Examen M1/Segmentation-Client-Marketing-IA/data/processed/sales_data_clean.csv
/media/Celesy20/M1 OCC/ML & DL/Examen M1/Segmentation-Client-Marketing-IA/data/processed/dataset_data_final.csv
